In [ ]:
# === Step 1: Import Necessary Libraries ===

import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
# Qiskit 1.0+ imports
from qiskit import QuantumCircuit
from qiskit.primitives import Sampler
from qiskit_aer import AerSimulator
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms.classifiers import QSVC
from IPython.display import display

# Random seed for reproducibility
seed = 128
np.random.seed(seed)

In [ ]:
# === Step 2: Load Processed Data ===
try:
    X_train = np.load("data/processed/X_train.npy")
    y_train = np.load("data/processed/y_train.npy")
    X_val = np.load("data/processed/X_val.npy")
    y_val = np.load("data/processed/y_val.npy")
    X_test = np.load("data/processed/X_test.npy")
    y_test = np.load("data/processed/y_test.npy")

    print("✅ Data loaded successfully.")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
except FileNotFoundError:
    raise SystemExit("❌ Processed data files not found. Please run preprocessing first.")


In [ ]:
# === Step 3: Define Amplitude Embedding Feature Map ===

def create_amplitude_embedding_circuit(feature_vector):
    """Create a quantum circuit that embeds the data using amplitudes."""
    feature_vector = np.array(feature_vector, dtype=float)
    norm = np.linalg.norm(feature_vector)
    if norm == 0:
        raise ValueError("Feature vector cannot be zero.")
    normalized = feature_vector / norm  # Normalize to unit length

    n_qubits = int(np.ceil(np.log2(len(normalized))))  # qubits = log2(vector_length)
    # Pad feature vector to match 2^n_qubits if needed
    if len(normalized) < 2**n_qubits:
        normalized = np.concatenate(
            [normalized, np.zeros(2**n_qubits - len(normalized))]
        )

    qc = QuantumCircuit(n_qubits)
    qc.initialize(normalized, range(n_qubits))
    return qc

# Example visualization
sample_vec = X_train[0]
qc_example = create_amplitude_embedding_circuit(sample_vec)
print(f"Feature map created with {qc_example.num_qubits} qubits.")
try:
    display(qc_example.draw("mpl", style="iqx"))
except Exception:
    print(qc_example)


In [ ]:
def amplitude_feature_map(x):
    """Helper: returns QuantumCircuit for each feature vector."""
    return create_amplitude_embedding_circuit(x)

sampler = Sampler(AerSimulator())
fidelity_kernel = FidelityQuantumKernel(
    feature_map=amplitude_feature_map, fidelity=sampler
)

print("\n✅ FidelityQuantumKernel (Amplitude Embedding) created successfully.")


In [ ]:
# === Step 5: Train the QSVM Model ===

qsvc = QSVC(quantum_kernel=fidelity_kernel)

print("\n🚀 Starting QSVM Training...")
start = time.time()
qsvc.fit(X_train, y_train)
print(f"✅ Training complete in {time.time() - start:.2f} seconds.")

# %%
# === Step 6: Evaluate the Model ===

print("\n🔍 Evaluating QSVM on test set...")
y_pred = qsvc.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\n🎯 QSVM Accuracy: {acc:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

# Visualization
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("QSVM Confusion Matrix (Amplitude Embedding)")
plt.show()

print("\n✅ Evaluation complete.")
